
# Hybridlane 2-Local Gadget & Heat-Equation Demo

This notebook assembles the continuous-variable (CV) / discrete-variable (DV) building blocks from
*Continuous LCU* to show how Hybridlane's operators can realize the 2-local gadget and reuse the same
compilation pattern for the discretized heat-equation generator from *Hybrid CV DV LCHS*.



## Environment Setup
Import Hybridlane alongside standard scientific tools and instantiate both a hybrid CV-DV device and a
reference qubit-only simulator.


In [1]:

import numpy as np
import pennylane as qml
import hybridlane as hqml

from dataclasses import dataclass
from typing import Callable, Dict, Optional, Tuple

from scipy.linalg import expm

# Hybrid CV-DV backend (bosonic qumode with Jaynes-Cummings interactions)
dev_hybrid = qml.device("bosonicqiskit.hybrid", max_fock_level=8)

# Reference qubit simulator for diagnostics
dev_qubit = qml.device("default.qubit", wires=2)



## Utility Layer: Hybrid Interactions & Swap Network
Helpers mirroring the compilation flow in Sec. IV of *Continuous LCU*, exposing:

- `HybridInteraction`: wraps a gate and its time-dependent parameters.
- `jaynes_cummings_interaction`: produces the mediated qubit-bus coupling.
- `apply_cv_dv_swap_network_trotter`: orchestrates mode/qubit swaps across a lattice.
- `apply_two_local_gadget`: implements the three-stroke sequence for a single ancilla mode.


In [2]:

@dataclass
class HybridInteraction:
    gate: Callable
    param_fn: Callable[[float], Tuple]
    kwargs: Optional[Dict[str, object]] = None

    def apply(self, dt: float, wires):
        params = self.param_fn(dt)
        kw = {} if self.kwargs is None else dict(self.kwargs)
        self.gate(*params, wires=wires, **kw)


def jaynes_cummings_interaction(coupling: float, phase: float = np.pi / 2) -> HybridInteraction:
    'Return a Jaynes-Cummings interaction with time-scaled strength.'

    def _params(dt: float, g=coupling, phi=phase):
        return (g * dt, phi)

    return HybridInteraction(gate=hqml.JaynesCummings, param_fn=_params)


def _resolve_mode_swap_gate(custom_gate=None):
    if custom_gate is not None:
        return custom_gate
    for candidate in ("CVSWAP", "CVSwap", "ModeSwap"):
        if hasattr(hqml, candidate):
            return getattr(hqml, candidate)
    return None


def _mode_swap_layer(logical, physical, swap_gate):
    'Rotate the logical-to-physical map for qumodes using CV SWAPs.'
    if len(physical) < 2:
        return logical
    for idx in range(len(physical) - 1):
        wires = [physical[idx], physical[idx + 1]]
        if swap_gate is not None:
            swap_gate(wires=wires)
        else:
            # Fallback beamsplitter-based SWAP (see Eq. 50 of Continuous LCU)
            qml.Beamsplitter(np.pi / 2, 0.0, wires=wires)
            qml.PhaseShift(np.pi, wires=wires[0])
            qml.PhaseShift(np.pi, wires=wires[1])
    return logical[1:] + logical[:1]


def _qubit_swap_layer(logical, physical):
    'Optionally permute qubits in lockstep with oscillator routing.'
    if len(physical) < 2:
        return logical
    for idx in range(len(physical) - 1):
        qml.SWAP(wires=[physical[idx], physical[idx + 1]])
    return logical[1:] + logical[:1]


def apply_cv_dv_swap_network_trotter(
    total_time: float,
    steps: int,
    qubit_wires,
    mode_wires,
    interactions: Dict[Tuple, HybridInteraction],
    *,
    qubit_labels=None,
    mode_labels=None,
    swap_qubits: bool = False,
    mode_swap=None,
):
    'Parallelized CV/DV Trotter step using the swap-network gadget.'

    dt = total_time / steps
    physical_qubits = list(qubit_wires)
    physical_modes = list(mode_wires)
    logical_qubits = list(qubit_labels or qubit_wires)
    logical_modes = list(mode_labels or mode_wires)
    swap_gate = _resolve_mode_swap_gate(mode_swap)
    layers = max(len(physical_modes), 1)

    for _ in range(steps):
        for _ in range(layers):
            max_pairs = min(len(physical_qubits), len(physical_modes))
            for idx in range(max_pairs):
                q_label = logical_qubits[idx]
                m_label = logical_modes[idx]
                term = interactions.get((q_label, m_label))
                if term is None:
                    continue
                term.apply(dt, [physical_qubits[idx], physical_modes[idx]])
            if swap_qubits:
                logical_qubits = _qubit_swap_layer(logical_qubits, physical_qubits)
            logical_modes = _mode_swap_layer(logical_modes, physical_modes, swap_gate)


def apply_two_local_gadget(
    total_time: float,
    steps: int,
    data_qubits,
    bus_mode,
    couplings: Dict[int, HybridInteraction],
):
    'Compile Eq. (36) via a shared bus mode in three strokes.'

    dt = total_time / steps
    ordering = list(data_qubits)

    for _ in range(steps):
        for q_wire in ordering:
            term = couplings[q_wire]
            term.apply(0.5 * dt, [q_wire, bus_mode])
        for q_wire in reversed(ordering):
            term = couplings[q_wire]
            term.apply(0.5 * dt, [q_wire, bus_mode])



## Example: Mediated Heisenberg Bond
Instantiate the gadget for a pair of qubits coupled through a single ancilla oscillator. The Jaynes-Cummings
interactions mediate `XX + YY + Δ ZZ` displacements depending on the chosen coupling constants.


In [3]:

qubits = [0, 1]
bus = "m0"

heisenberg_couplings = {
    0: jaynes_cummings_interaction(0.8),
    1: jaynes_cummings_interaction(0.65),
}

@qml.qnode(dev_hybrid)
def two_local_gadget_demo(total_time: float, steps: int):
    qml.X(qubits[0])
    qml.RY(0.35, wires=qubits[1])
    apply_two_local_gadget(
        total_time=total_time,
        steps=steps,
        data_qubits=qubits,
        bus_mode=bus,
        couplings=heisenberg_couplings,
    )
    return (
        qml.expval(qml.PauliZ(qubits[0])),
        qml.expval(qml.PauliZ(qubits[1])),
        hqml.expval(hqml.NumberOperator(bus)),
    )

response = two_local_gadget_demo(total_time=0.9, steps=4)
print("<Z0>, <Z1>, <n_bus> =", response)


<Z0>, <Z1>, <n_bus> = (array(-0.15180329), array(0.88555329), array(0.39718864))



## Heat-Equation Generator (Section 3.4.2)
Reproduce the discretized 1D heat equation matrix \(L\) and decompose it into Pauli words. These coefficients
feed into a control schedule that reuses the two-local gadget alongside single-qubit rotations, mirroring the
LCU-based compilation of the paper.


In [4]:

SX = np.array([[0, 1], [1, 0]], dtype=complex)
SY = np.array([[0, -1j], [1j, 0]], dtype=complex)
I2 = np.eye(2, dtype=complex)

PAULI_BASIS = {
    "I⊗I": np.kron(I2, I2),
    "X⊗I": np.kron(SX, I2),
    "I⊗X": np.kron(I2, SX),
    "X⊗X": np.kron(SX, SX),
    "Y⊗Y": np.kron(SY, SY),
}


def heat_equation_matrix(n_interior: int = 4, h: float = 1.0) -> np.ndarray:
    diag = 2 * np.ones(n_interior)
    off = -1 * np.ones(n_interior - 1)
    mat = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)
    return mat / (h**2)


def pauli_decomposition(matrix: np.ndarray) -> Dict[str, float]:
    A = np.stack([op.reshape(-1) for op in PAULI_BASIS.values()], axis=1).real
    coeffs, *_ = np.linalg.lstsq(A, matrix.reshape(-1).real, rcond=None)
    return {label: coeff for label, coeff in zip(PAULI_BASIS.keys(), coeffs)}

L_matrix = heat_equation_matrix()
L_coeffs = pauli_decomposition(L_matrix)

print("Heat-equation L coefficients (Pauli basis):")
for label, coeff in L_coeffs.items():
    print(f"  {label}: {coeff:+.4f}")


Heat-equation L coefficients (Pauli basis):
  I⊗I: +2.0000
  X⊗I: +0.0000
  I⊗X: -1.0000
  X⊗X: -0.5000
  Y⊗Y: -0.5000



### Scheduling the Heat-Equation Evolution
Map the coefficients onto a control sequence that combines:

- single-qubit `X` rotations (for the linear terms), and
- the CV-mediated two-qubit block (for the `XX + YY` sector).

The example below generates a quantum tape so the gate structure can be inspected before execution.


In [5]:

def apply_heat_equation_trotter(
    total_time: float,
    steps: int,
    data_qubits,
    bus_mode,
    coeffs: Dict[str, float],
):
    dt = total_time / steps
    single_terms = {
        data_qubits[0]: coeffs.get("X⊗I", 0.0),
        data_qubits[1]: coeffs.get("I⊗X", 0.0),
    }
    exchange_coeff = coeffs.get("X⊗X", 0.0)

    for _ in range(steps):
        for qubit, coeff in single_terms.items():
            if abs(coeff) > 1e-9:
                qml.RX(2 * coeff * dt, wires=qubit)
        if abs(exchange_coeff) > 1e-9:
            mediated = {
                data_qubits[0]: jaynes_cummings_interaction(np.sqrt(abs(exchange_coeff))),
                data_qubits[1]: jaynes_cummings_interaction(np.sqrt(abs(exchange_coeff))),
            }
            apply_two_local_gadget(
                total_time=dt,
                steps=1,
                data_qubits=data_qubits,
                bus_mode=bus_mode,
                couplings=mediated,
            )


with qml.tape.QuantumTape() as heat_tape:
    apply_heat_equation_trotter(
        total_time=0.25,
        steps=2,
        data_qubits=qubits,
        bus_mode=bus,
        coeffs=L_coeffs,
    )

print("Scheduled operations for one Trotter step:")
for op in heat_tape.operations:
    print("  ", op)


Scheduled operations for one Trotter step:
   RX(np.float64(-0.2500000000000001), wires=[1])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[0, 'm0'])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[1, 'm0'])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[1, 'm0'])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[0, 'm0'])
   RX(np.float64(-0.2500000000000001), wires=[1])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[0, 'm0'])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[1, 'm0'])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[1, 'm0'])
   JaynesCummings(np.float64(0.04419417382415922), 1.5707963267948966, wires=[0, 'm0'])



### Optional: Hybrid Simulation Snapshot
Run the schedule on the hybrid device for a short evolution time to inspect hybrid observables. This remains unitary,
corresponding to a block-encoding-style primitive rather than the full non-unitary diffusion.


In [6]:

@qml.qnode(dev_hybrid)
def heat_equation_demo(total_time: float, steps: int):
    qml.Hadamard(qubits[0])
    qml.RY(0.2, wires=qubits[1])
    apply_heat_equation_trotter(
        total_time=total_time,
        steps=steps,
        data_qubits=qubits,
        bus_mode=bus,
        coeffs=L_coeffs,
    )
    return (
        qml.expval(qml.PauliX(qubits[0])),
        qml.expval(qml.PauliX(qubits[1])),
        hqml.expval(hqml.NumberOperator(bus)),
    )

snapshot = heat_equation_demo(total_time=0.1, steps=2)
print("Hybrid expectation snapshot (<X0>, <X1>, <n_bus>):", snapshot)


Hybrid expectation snapshot (<X0>, <X1>, <n_bus>): (array(0.99750077), array(0.19576491), array(0.00306345))



### Reference: Classical Diffusion Step
Compare the qubit-only expectation value obtained from directly applying \(e^{-t L}\) to the same initial state.
This is a classical benchmark (no qumode) to keep tabs on the diffusion dynamics.


In [7]:

initial_state = np.array([1, 0, 0, 0], dtype=complex)
L = L_matrix

def classical_diffusion_expectation(time: float, state: np.ndarray) -> float:
    evolved = expm(-time * L) @ state
    evolved /= np.linalg.norm(evolved)
    obs = PAULI_BASIS["X⊗I"]
    return float(np.real(np.vdot(evolved, obs @ evolved)))

benchmark = classical_diffusion_expectation(0.1, initial_state)
print("Classical diffusion <X⊗I> after Δt=0.1:", benchmark)


Classical diffusion <X⊗I> after Δt=0.1: 0.009909215780794565
